# E0: legacy provenance and cost audit

This notebook recomputes observations from real saved legacy logs; it does not rerun inference. Corrected W3/W4 are separate from original W3/W4. Empty saved answers remain empty and are scored by official rules, not repaired. Communication counts in these files remain **word proxies**; provider input/output usage is unknown. The current source checksum is not proof of the historical executing source revision.

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd() if (Path.cwd() / 'protocol.py').exists() else Path.cwd() / 'WEBExtension'
if not (ROOT / 'protocol.py').exists():
    raise FileNotFoundError('Launch Jupyter from the project root or WEBExtension folder')
ROOT = ROOT.resolve()
sys.path.insert(0, str(ROOT))
from protocol import load_official, write_json, file_hash
from experiments import load_config, preview, freeze, run
CFG = load_config(ROOT / 'config.json')
EXECUTE = False  # Set True only after configuring your real backend and maximum call budget.
print('Notebook root:', ROOT)
print('Inference enabled:', EXECUTE)


Notebook root: /home/saurab/Documents/projectRLMAIN/WEBExtension
Inference enabled: False


In [2]:
from audit import audit_legacy
summary, checks = audit_legacy()
display(summary)
print('Recreated tables:', ROOT/'artifacts/legacy_audit')

,experiment,arm,n,F1,EM,empty_answers,mean_word_proxy
0,W1_web_cost_accuracy,MAD,100,48.362671,30.0,6,296.98
1,W1_web_cost_accuracy,NoComm,100,46.794372,31.0,12,0.00
2,W1_web_cost_accuracy,SPARSECOMM,100,44.511039,29.0,11,44.94
3,W2_context_rationing,Full_Transmission,100,47.152065,29.0,9,75.65
4,W2_context_rationing,Random_Truncation,100,44.485398,27.0,11,66.86
5,W2_context_rationing,Semantic_Rationing,100,46.568731,29.0,11,66.25
6,W3_adaptive_stopping,Clean_Adaptive,100,43.636264,27.0,14,90.97
7,W3_adaptive_stopping,Clean_Fixed,100,48.488991,30.0,10,88.87
8,W3_adaptive_stopping,Noisy_Adaptive,100,52.035920,36.0,13,96.24
9,W3_adaptive_stopping,Noisy_Fixed,100,53.499556,37.0,10,95.68


Recreated tables: /home/saurab/Documents/projectRLMAIN/WEBExtension/artifacts/legacy_audit


In [3]:
display([{'experiment':c['experiment'],'matching_proxy_violations':c.get('matching_proxy_violations'),
          'n_questions':c.get('n_questions'),'limitations':c.get('limitations')} for c in checks])
print('W4 matching excludes Full broadcast but includes the arm labelled Matched.')

[{'experiment': 'W1_web_cost_accuracy',
  'matching_proxy_violations': None,
  'n_questions': 100,
  'limitations': ['Historical source revision not established by hashes of current files',
   'Raw per-call prompt/response and separate provider usage not present in raw_results',
   'Measured answer scores do not authenticate historical backend execution']},
 {'experiment': 'W2_context_rationing',
  'matching_proxy_violations': None,
  'n_questions': 100,
  'limitations': ['Historical source revision not established by hashes of current files',
   'Raw per-call prompt/response and separate provider usage not present in raw_results',
   'Measured answer scores do not authenticate historical backend execution']},
 {'experiment': 'W3_adaptive_stopping',
  'matching_proxy_violations': None,
  'n_questions': 100,
  'limitations': ['Historical source revision not established by hashes of current files',
   'Raw per-call prompt/response and separate provider usage not present in raw_results',


W4 matching excludes Full broadcast but includes the arm labelled Matched.


## Interpretation

Matching must be checked per question, not only on the mean. The corrected sparse/static/random arms and corrected matched-broadcast arm should also be checked separately. Exact prompt and provider usage reconstruction is unavailable from `raw_results.jsonl`; do not turn these proxy counts into prices or total inference tokens. Audit manifests contain claimed source revisions, artifact paths and checksums, plus unresolved limitations.

In [4]:
import pandas as pd
q = pd.read_csv(ROOT/'artifacts/legacy_audit/question_level_rescore.csv')
w4=q[q.experiment=='W4_corrected'].pivot(index='id',columns='arm',values='word_proxy_comm')
core=['Static_Role','Learned_SparseComm','Random_Routing']
print('Core corrected routing proxy mismatches:',int((w4[core].nunique(axis=1)>1).sum()))
print('Matched broadcast versus static proxy mismatches:',int((w4.Complete_Broadcast_Matched!=w4.Static_Role).sum()))
print('Saved empty answers by experiment:',q.groupby('experiment').empty_answer.sum().to_dict())

Core corrected routing proxy mismatches: 0
Matched broadcast versus static proxy mismatches: 45
Saved empty answers by experiment: {'W1_web_cost_accuracy': 29, 'W2_context_rationing': 31, 'W3_adaptive_stopping': 47, 'W3_corrected': 43, 'W4_corrected': 76, 'W4_counterfactual_routing': 35}
